In [ ]:
# -*- coding: utf-8 -*-
"""FutureEnginner - Block Training.ipynb"""

import os

# ── Step 1: Extract dataset (manually upload dataset.zip first) ──
import zipfile
with zipfile.ZipFile('/content/dataset.zip', 'r') as z:
    z.extractall('/content')

print("Dataset extracted!")
!find /content/dataset -type d

# ── Step 2: Verify labels ──
print(f"Train images: {len(os.listdir('/content/dataset/images/train'))}")
print(f"Val images:   {len(os.listdir('/content/dataset/images/val'))}")
print(f"Train labels: {len(os.listdir('/content/dataset/labels/train'))}")
print(f"Val labels:   {len(os.listdir('/content/dataset/labels/val'))}")

sample = os.listdir('/content/dataset/labels/train')[0]
with open(f'/content/dataset/labels/train/{sample}') as f:
    print(f"\nSample label ({sample}):")
    print(f.read())

# ── Step 3: Install ultralytics ──
!pip install ultralytics -q

# ── Step 4: Train ──
from ultralytics import YOLO

model = YOLO('yolo26s.pt')

model.train(
    data='/content/dataset/dataset.yaml',   # now nc=3: green, red, magenta
    epochs=200,
    imgsz=224,
    batch=16,
    name='wro_detector',
    patience=20,
    device=0,
    augment=True,
    hsv_h=0.015,         # hue shift — critical for color detection
    hsv_s=0.7,           # saturation shift
    hsv_v=0.4
)

# ── Step 5: Validate ──
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

# ── Step 6: Export to ONNX ──
model = YOLO('/content/runs/detect/wro_detector/weights/best.pt')
model.export(format='onnx', imgsz=224, dynamic=True)

# ── Step 7: Download ──
from google.colab import files
files.download('/content/runs/detect/wro_detector/weights/best.onnx')

with open('/content/labelmap.txt', 'w') as f:
    f.write('green\nred\nmagenta\n')
files.download('/content/labelmap.txt')

print("\nDone! Files downloaded: best.onnx, labelmap.txt")

Dataset extracted!
/content/dataset
/content/dataset/images
/content/dataset/images/val
/content/dataset/images/train
/content/dataset/labels
/content/dataset/labels/val
/content/dataset/labels/train
Train images: 228
Val images:   57
Train labels: 228
Val labels:   57

Sample label (red_exp300_wb4500_0053.txt):
1 0.742969 0.328125 0.051562 0.122917
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment,

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done! Files downloaded: best.onnx, labelmap.txt
